# GPU transformer benchmark

This Colab workflow compares a compact DistilBERT model with the classical baseline on the **same five connected-group outer folds**. Enable a GPU runtime before running it. The command writes fold-level JSON evidence; it does not overwrite the published classical benchmark.

Expected cost on a free GPU is hardware- and quota-dependent. Start with two folds for a smoke test, then run all five for publishable evidence. Free Colab sessions can disconnect, so download the JSON result immediately.

In [ ]:
!git clone https://github.com/kristishqau/SentimentAnalysis_NLP.git
%cd SentimentAnalysis_NLP
%pip install -q -e ".[transformer]" "scikit-learn==1.9.1"

In [ ]:
import torch

assert torch.cuda.is_available(), "Select Runtime → Change runtime type → GPU"
print(torch.cuda.get_device_name(0))

## Smoke test

Two grouped folds verify the environment. This is not the final comparison.

In [ ]:
!sentiment-transformer-benchmark \
  --folds 2 --epochs 1 --batch-size 32 \
  --output artifacts/transformer_smoke.json

## Publishable five-fold comparison

Run these only after the smoke test succeeds. Both models use identical canonical outer folds, grouped inner validation, macro-F1 checkpoint selection, and balanced loss. If memory is limited, lower `--batch-size` to 16.

In [ ]:
!sentiment-transformer-benchmark \
  --folds 5 --epochs 3 --batch-size 32 \
  --validation-folds 5 --early-stopping-patience 1 \
  --class-weighting balanced \
  --reference reports/benchmark.json \
  --output reports/transformer_distilbert_benchmark.json

In [ ]:
!sentiment-transformer-benchmark \
  --model cardiffnlp/twitter-roberta-base \
  --folds 5 --epochs 3 --batch-size 32 \
  --validation-folds 5 --early-stopping-patience 1 \
  --class-weighting balanced \
  --reference reports/benchmark.json \
  --output reports/transformer_benchmark.json

In [ ]:
import json
from pathlib import Path

distilbert = json.loads(Path("reports/transformer_distilbert_benchmark.json").read_text())
twitter_roberta = json.loads(Path("reports/transformer_benchmark.json").read_text())
classical = json.loads(Path("reports/benchmark.json").read_text())
print("DistilBERT macro-F1:", distilbert["summary"]["macro_f1"])
print("Twitter-RoBERTa macro-F1:", twitter_roberta["summary"]["macro_f1"])
print("Classical macro-F1:", classical["candidates"]["entity_word_char"]["summary"]["macro_f1"])

In [ ]:
from google.colab import files

files.download("reports/transformer_distilbert_benchmark.json")
files.download("reports/transformer_benchmark.json")